# Estimation with the SDK Engine

Run Kriging, Inverse Distance Weighting (IDW) and K-Nearest Neighbors (KNN) using the SDK's parameter models and typed-object APIs. The main workflow is the familiar `tasks.run(...)` call, which already executes through the SDK engine.

Before running, populate a workspace with the composites, variogram and target block model. The source needs `Cu_pct`, `Geological_model` and `IDW declus`; the target needs `Geological model`.

These tasks write `Cu_OK`, `Cu_IDW`, `Cu_KNN` and `KV` onto the target block model. Running both execution sections updates those same columns twice; `preview=True` enables preview tasks, not a dry run.

In [ ]:
import asyncio

import evo.objects.typed as objs
import numpy as np
import pandas as pd
import plotly.graph_objects as go
from evo.notebooks import ServiceManagerWidget

from evo.compute import ComputeClient, tasks
from evo.compute.tasks.geostatistics import idw, knn, kriging

## Authenticate

Authenticate with the existing SDK widget and select the workspace containing the example objects. Replace `your-client-id` with your registered application ID.

In [ ]:
evo_connection = await ServiceManagerWidget.with_auth_code(
    client_id="your-client-id", cache_location="./notebook-data"
).login()

In [ ]:
%load_ext evo.widgets

## Load the Objects Needed

Load the same composites, variogram and block model through the existing typed-object API. Object references contain metadata; use `.to_dataframe()` when numerical data is needed.

In [ ]:
comps = await objs.object_from_path(evo_connection, "WP comps.json")
variogram = await objs.object_from_path(evo_connection, "Cu_pct Variogram Model.json")
block_model = await objs.object_from_path(evo_connection, "WP Regular BlockModel.json")

Inspect the objects using their usual notebook representations, including links to the Portal and Viewer.

In [ ]:
variogram

In [ ]:
comps.attributes

In [ ]:
block_model

## Generate the Estimation Parameters

Keep the existing parameter models. Scale the variogram ellipsoid by 1.5 and share its search neighborhood and Early Diorite filters across all three estimations.

In [ ]:
ell = variogram.get_ellipsoid().scaled(1.5)
srch = tasks.SearchNeighborhood(ellipsoid=ell, max_samples=30, min_samples=1)

In [ ]:
early_diorite_source_filter = tasks.Filter(
    where=tasks.FilterCondition(attribute=comps.attributes["Geological_model"], operator="in", values=["Early Diorite"])
)

early_diorite_target_filter = tasks.Filter(
    where=tasks.FilterCondition(
        attribute=block_model.attributes["Geological model"], operator="in", values=["Early Diorite"]
    )
)

### Kriging

Estimate `Cu_OK` using 5 x 5 x 2 block discretisation and write kriging variance to `KV`.

In [ ]:
krig_params = kriging.KrigingParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_OK"],
    target_filter=early_diorite_target_filter,
    variogram=variogram,
    neighborhood=srch,
    block_discretization=kriging.BlockDiscretization(nx=5, ny=5, nz=2),
    diagnostics=kriging.KrigingDiagnostics(kriging_variance=block_model.attributes["KV"]),
)

The existing `KrigingDiagnostics` API is unchanged. A field accepts `True` for its recommended name, a string for a chosen name, or a target attribute. Each diagnostic must use a different output name.

### Inverse Distance Weighting (IDW)

In [ ]:
idw_params = idw.IDWParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_IDW"],
    target_filter=early_diorite_target_filter,
    neighborhood=srch,
    power=2,
)

### K-Nearest Neighbors (KNN)

Use the same SDK neighborhood configuration. `KNNParameters` does not expose a separate `k` argument.

In [ ]:
knn_params = knn.KNNParameters(
    source=comps.attributes["Cu_pct"],
    source_filter=early_diorite_source_filter,
    target=block_model.attributes["Cu_KNN"],
    target_filter=early_diorite_target_filter,
    neighborhood=srch,
)

## Execute with the Familiar SDK Call

Use `tasks.run(...)` to execute the parameter-model batch through the engine. The SDK shares catalogue discovery and retains ordered native results and aggregate progress feedback.

In [ ]:
# Existing parameter models and the batch call stay the same.
results = await tasks.run(evo_connection, [krig_params, idw_params, knn_params], preview=True)
results

## View the Results

Refresh the block model and query the result columns using the same typed-object methods as before. Keep this dataframe in memory for comparison after the explicit engine run.

In [ ]:
updated_block_model = await block_model.refresh()
updated_block_model

In [ ]:
estimate_columns = ["Cu_KNN", "Cu_IDW", "Cu_OK"]
result_columns = [*estimate_columns, "KV", "Geological model"]
block_model_df = await updated_block_model.to_dataframe(columns=result_columns)
block_model_df.head()

In [ ]:
block_model_df[[*estimate_columns, "KV"]].describe()

## Optional: Submit and Manage Jobs

For normal estimation, keep using `await tasks.run(...)` above. That call already uses the SDK engine: it submits the batch concurrently, waits for completion, and returns ordered native SDK results with aggregate progress feedback. You do not need to create a `ComputeClient` or run this section to use the engine.

Explicit submission is useful when you need control over individual jobs between submission and completion. Each task's `submit(...)` method returns a `TaskJob` without waiting for the computation to finish, so you can:

- Record each job's ID and URL for tracking.
- Inspect an individual job with `await job.status()`.
- Cancel a job with `await job.cancel()`.
- Separate submission from collecting the final outputs with `await job.results()`.

Use this interface when you need those job-management features, not as a replacement for the familiar batch workflow. It uses the same authenticated connection and validation. Kriging accepts the typed SDK inputs directly and returns its usual `KrigingResult`, including dataframe and object helpers. IDW and KNN use schema-shaped inputs and return generic hydrated results.

Set progress feedback on `ComputeClient(..., fb=feedback)` or `job.results(fb=feedback)`. It controls the local wait, so it is not a task keyword. Save a job URL to reopen it later with `await TaskJob.from_url(evo_connection, url, preview=True)`; preview must be opted into again because the URL does not record it.

This demonstration submits the same three estimations again and writes the same output columns a second time. `preview=True` is not a dry run. The comparison and plotting cells below use this second run's `engine_df` to demonstrate parity; that notebook dependency is not a requirement for using the engine. A normal estimation workflow can work directly with the `block_model_df` produced by `tasks.run(...)` above.

In [ ]:
# The engine uses the same connection; no new authentication is needed.
client = ComputeClient(evo_connection)
estimation_source = {
    "object": idw_params.source.object,
    "attribute": idw_params.source.attribute,
    "filter": early_diorite_source_filter,
}

In [ ]:
# Submit all three tasks concurrently without waiting for their results.
jobs = await asyncio.gather(
    client.geostatistics.kriging.submit(
        source=krig_params.source,
        target=krig_params.target,
        variogram=krig_params.variogram,
        neighborhood=krig_params.search,
        kriging_method=krig_params.method,
        source_filter=krig_params.source_filter,
        target_filter=krig_params.target_filter,
        block_discretization=krig_params.block_discretization,
        diagnostics=krig_params.diagnostics,
        preview=True,
    ),
    client.geostatistics.idw.submit(
        source=estimation_source,
        target={
            "object": idw_params.target.object,
            "attribute": idw_params.target.attribute,
            "filter": early_diorite_target_filter,
        },
        neighborhood=srch,
        power=idw_params.power,
        preview=True,
    ),
    client.geostatistics.knn.submit(
        source=estimation_source,
        target={
            "object": knn_params.target.object,
            "attribute": knn_params.target.attribute,
            "filter": early_diorite_target_filter,
        },
        neighborhood=srch,
        preview=True,
    ),
)

In [ ]:
# Handles expose IDs and URLs; status() and cancel() are also available.
pd.DataFrame({"Task": [job.task for job in jobs], "Job ID": [job.id for job in jobs], "URL": [job.url for job in jobs]})

Collect the results when ready. `asyncio.gather` preserves task order. If a job does not need to be retained, `client.arun(topic, task, parameters)` combines submission and waiting in one call.

In [ ]:
# Wait for all jobs on the notebook's existing event loop.
engine_results = await asyncio.gather(*(job.results() for job in jobs))
engine_results[0]

Kriging submission keeps the native result helpers. Load the target with `get_target_object()`, then reuse the existing dataframe query. Generic results from the other tasks also provide loadable target references.

In [ ]:
engine_block_model = await engine_results[0].get_target_object()
engine_df = await engine_block_model.to_dataframe(columns=result_columns)
engine_df.head()

## Compare the Two Runs

Compare the saved SDK dataframe with the engine dataframe for the same block rows, estimates and variance. The match column uses relative tolerance `1e-9` and absolute tolerance `1e-12`, allowing small floating-point differences.

In [ ]:
comparison_columns = [*estimate_columns, "KV"]
comparison_df = pd.DataFrame(
    {
        "SDK mean": block_model_df[comparison_columns].mean(),
        "Engine mean": engine_df[comparison_columns].mean(),
        "Max absolute difference": (block_model_df[comparison_columns] - engine_df[comparison_columns]).abs().max(),
        "Values match": [
            block_model_df.index.equals(engine_df.index)
            and block_model_df[column].notna().any()
            and np.allclose(block_model_df[column], engine_df[column], rtol=1e-9, atol=1e-12, equal_nan=True)
            for column in comparison_columns
        ],
    }
)
comparison_df

## Compare Estimates with the Composites

Filter the composites and estimates to Early Diorite and apply the composite declustering weights. Load the composite data, combine it with the engine estimates, and inspect their distributions and summary statistics.

In [ ]:
comps = await comps.refresh()
comps_df = await comps.to_dataframe("Cu_pct", "Geological_model", "IDW declus")
comps_df.head()

In [ ]:
composites_df = comps_df.query("Geological_model == 'Early Diorite'")[["Cu_pct", "IDW declus"]].rename(
    columns={"Cu_pct": "Cu (%)", "IDW declus": "weight"}
)
composites_df = composites_df.dropna().query("weight > 0")
composites_df["Method"] = "Declustered Composites"

estimates_df = (
    engine_df.query("`Geological model` == 'Early Diorite'")[estimate_columns]
    .melt(var_name="Method", value_name="Cu (%)")
    .dropna()
)
estimates_df["weight"] = 1.0

histogram_df = pd.concat([composites_df, estimates_df], ignore_index=True)
histogram_df.head()

In [ ]:
# Plotly can render the weighted comparison without external notebook helpers.
methods = ["Declustered Composites", "Cu_KNN", "Cu_IDW", "Cu_OK"]
colors = ["#071E22", "#1D7874", "#F4C095", "#EE2E31"]
fig = go.Figure()
for method, color in zip(methods, colors, strict=True):
    values = histogram_df[histogram_df["Method"] == method]
    fig.add_trace(
        go.Histogram(
            x=values["Cu (%)"],
            y=values["weight"],
            histfunc="sum",
            histnorm="probability density",
            bingroup="grades",
            name=method,
            marker_color=color,
            opacity=0.6,
        )
    )
fig.update_layout(
    title="Early Diorite: Declustered Composites vs Estimation Results",
    barmode="overlay",
    xaxis_title="Cu (%)",
    yaxis_title="Density",
    template="plotly_white",
)
fig.show()

Inspect the weighted composite statistics alongside the estimation statistics. The estimate variance below is the population variance across estimated blocks; it is distinct from the per-block kriging variance in `KV`.

In [ ]:
composite_values = composites_df["Cu (%)"]
composite_weights = composites_df["weight"]
weighted_mean = np.average(composite_values, weights=composite_weights)
composite_summary = pd.Series(
    {
        "count": len(composite_values),
        "min": composite_values.min(),
        "mean": weighted_mean,
        "variance": np.average((composite_values - weighted_mean) ** 2, weights=composite_weights),
        "max": composite_values.max(),
    },
    name="Declustered Composites",
)
estimate_summary = (
    estimates_df.groupby("Method")["Cu (%)"]
    .agg(count="count", min="min", mean="mean", variance=lambda values: values.var(ddof=0), max="max")
    .T
)
summary_df = pd.concat([composite_summary, estimate_summary], axis=1).reindex(columns=methods)
summary_df

## Close the Connection

After the computations and data downloads finish, release the connection opened by widget login on the same notebook event loop.

In [ ]:
await evo_connection.get_connector().close()